In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Netflix Recommendation System") \
    .getOrCreate()

print("Spark Ready")

Spark Ready


In [2]:
movies = spark.read.csv(
    r"D:\GitHub\DPySpark\data\movies.csv",
    header=True,
    inferSchema=True
)

ratings = spark.read.csv(
    r"D:\GitHub\DPySpark\data\ratings.csv",
    header=True,
    inferSchema=True
)

In [3]:
ratings_small = ratings.limit(500000)

ratings_small.count()

500000

In [4]:
from pyspark.ml.recommendation import ALS

In [5]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,
    maxIter=5,
    regParam=0.1,
    coldStartStrategy="drop"
)

In [6]:
model = als.fit(ratings_small)

print("Model Trained")

Model Trained


In [17]:
recommendations = model.recommendForAllUsers(5)

In [18]:
recommendations.show(10, truncate=False)

+------+--------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                         |
+------+--------------------------------------------------------------------------------------------------------+
|1     |[{33819, 5.6741686}, {3881, 5.515375}, {178771, 5.328937}, {168760, 5.328937}, {140245, 5.328937}]      |
|12    |[{3881, 5.2811537}, {65709, 5.129778}, {60103, 5.0934334}, {69699, 5.0874796}, {117364, 5.0798593}]     |
|22    |[{117364, 6.5534854}, {6653, 6.515445}, {60103, 6.2922516}, {7193, 6.138723}, {185571, 6.112522}]       |
|26    |[{117364, 5.2704926}, {3881, 5.0732203}, {60103, 5.0632415}, {95973, 5.0250483}, {127019, 5.024698}]    |
|27    |[{117364, 5.781625}, {60103, 5.767655}, {189475, 5.63783}, {107700, 5.63783}, {91808, 5.63783}]         |
|28    |[{117364, 6.649982}, {69699, 6.6275473}, {3881, 6.5370364}, {127019, 6.529359}, 

In [19]:
from pyspark.sql.functions import explode

user_movies = recommendations.select(
    "userId",
    explode("recommendations").alias("rec")
)

user_movies = user_movies.select(
    "userId",
    user_movies.rec.movieId.alias("movieId"),
    user_movies.rec.rating.alias("predicted_rating")
)

In [20]:
user_movies.show(10, truncate=False)

+------+-------+----------------+
|userId|movieId|predicted_rating|
+------+-------+----------------+
|1     |33819  |5.6741686       |
|1     |3881   |5.515375        |
|1     |178771 |5.328937        |
|1     |168760 |5.328937        |
|1     |140245 |5.328937        |
|12    |3881   |5.2811537       |
|12    |65709  |5.129778        |
|12    |60103  |5.0934334       |
|12    |69699  |5.0874796       |
|12    |117364 |5.0798593       |
+------+-------+----------------+
only showing top 10 rows


In [21]:
final_recommendations = user_movies.join(
    movies,
    on="movieId",
    how="left"
)

In [22]:
final_recommendations.show(20, truncate=False)

+-------+------+----------------+-----------------------------------------------------------+---------------------+
|movieId|userId|predicted_rating|title                                                      |genres               |
+-------+------+----------------+-----------------------------------------------------------+---------------------+
|33819  |1     |5.6741686       |Heights (2004)                                             |Drama                |
|3881   |1     |5.515375        |Phish: Bittersweet Motel (2000)                            |Documentary          |
|178771 |1     |5.328937        |Adele Hasn't Had Her Dinner Yet (1978)                     |Comedy|Crime         |
|168760 |1     |5.328937        |Cosy Dens (1999)                                           |Comedy|Drama         |
|140245 |1     |5.328937        |Twenty (2015)                                              |Comedy|Drama         |
|3881   |12    |5.2811537       |Phish: Bittersweet Motel (2000)        

In [23]:
from pyspark.sql.functions import least, lit

final_recommendations = final_recommendations.withColumn(
    "display_rating",
    least("predicted_rating", lit(5.0))
)

In [24]:
final_recommendations.show(20, truncate=False)

+-------+------+----------------+-----------------------------------------------------------+---------------------+--------------+
|movieId|userId|predicted_rating|title                                                      |genres               |display_rating|
+-------+------+----------------+-----------------------------------------------------------+---------------------+--------------+
|33819  |1     |5.6741686       |Heights (2004)                                             |Drama                |5.0           |
|3881   |1     |5.515375        |Phish: Bittersweet Motel (2000)                            |Documentary          |5.0           |
|178771 |1     |5.328937        |Adele Hasn't Had Her Dinner Yet (1978)                     |Comedy|Crime         |5.0           |
|168760 |1     |5.328937        |Cosy Dens (1999)                                           |Comedy|Drama         |5.0           |
|140245 |1     |5.328937        |Twenty (2015)                                     